In [ ]:
import torch.nn as nn
import torch
import torchvision
import torchvision.transforms.v2 as T
import numpy as np

toTensor = T.Compose([T.ToImage() , T.ToDtype(torch.float32 , scale=True)])

train_and_valid_data = torchvision.datasets.FashionMNIST(root="datasets" , download=True , train=True , transform=toTensor)
test_data = torchvision.datasets.FashionMNIST(root="datasets" , download=True , train=False , transform=toTensor)

100%|██████████| 26.4M/26.4M [00:08<00:00, 3.07MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 143kB/s]
100%|██████████| 4.42M/4.42M [00:01<00:00, 2.47MB/s]
100%|██████████| 5.15k/5.15k [00:00<?, ?B/s]


In [3]:
torch.manual_seed(42)
train_data , valid_data = torch.utils.data.random_split(train_and_valid_data , [55000 , 5000])

In [4]:
from torch.utils.data import DataLoader

train_data_loader = DataLoader(train_data , batch_size=32 , shuffle=True )
test_data_loader = DataLoader(test_data , batch_size = 32)
valid_data_loader = DataLoader(valid_data , batch_size= 32)

In [ ]:
#Classifier
class imageClassifier(nn.Module):
    def __init__(self, n_inputs , n_classes):
        super().__init__()
        self.model = nn.Sequential(
            nn.Flatten(),
            nn.Linear(n_inputs , 300),
            nn.ReLU(),
            nn.Linear(300, 200),
            nn.ReLU(),
            nn.Linear(200 , n_classes)
        )
    def forward(self , x):
        return self.model(x)
    
    def train(model , criterion , optimizer , data_loader , n_epochs):
        model.train()
        for epoch in range(n_epochs):
            total_loss = 0.0
            for x_batch, y_batch in data_loader:
                x_batch , y_batch = x_batch.to(device = "cuda") , y_batch.to(device = "cuda")
                y_pred = model(x_batch)
                loss = criterion-(y_pred , y_batch)
                total_loss += loss.item()
                loss.backward()
                optimizer.step()
                optimizer.zero_grad()
            mean_loss = total_loss / len(train_data_loader)
            print(f"Epoch{epoch + 1} loss :{mean_loss}")
    


In [ ]:
torch.manual_seed(42)
model  = imageClassifier(n_inputs=28*28 , n_classes=10)
entropy_class = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD()
